# CrisisMMD preliminary experiment

Feasibility check for the Project 2.1 proposal. Frozen EfficientNet-B0 (images) and BERT (text) features are extracted once, then logistic regression compares image only, text only, early fusion (concatenated features) and late fusion (weighted mean of the two unimodal probabilities) on the agreed label splits of the informative and humanitarian tasks.

Data is not included. Download CrisisMMD v2.0 and the data splits from https://crisisnlp.qcri.org/crisismmd and accept the CrisisNLP terms of use. Expected layout: `../data/CrisisMMD_v2.0/data_image/` and `../data/splits/crisismmd_datasplit_all/` and `../data/splits/crisismmd_datasplit_agreed_label/`.

Dependencies: Python 3.13, torch 2.8, torchvision 0.23, transformers 4.57, scikit-learn 1.7, pandas 2.3, numpy 2.2, pillow. Run on an Apple M3 GPU (MPS); the code falls back to CPU.

## 1. Feature extraction

Tweets are cleaned first: broken UTF-8 encoding is repaired, the retweet prefix and URLs are removed and user mentions become `@user`. Images are resized to 256 and centre cropped to 224. Features are stored as float16 in `../cache/`, which stays local.

In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

In [2]:
TF = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])])


class Images(Dataset):
    """Load and transform images from a list of paths."""

    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return TF(im.convert("RGB"))


def fix_mojibake(text):
    """Undo UTF-8 text that was decoded as cp1252, when that is the case."""
    if "â" not in text and "Ã" not in text:
        return text
    try:
        return text.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


def clean_text(text):
    """Fix encoding, drop the RT prefix, mask URLs and user mentions."""
    text = fix_mojibake(text)
    text = re.sub(r"^RT @\w+:\s*", "", text)
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"@\w+", "@user", text)
    return re.sub(r"\s+", " ", text).strip()


def load_index(splits_dir):
    """All image rows of the humanitarian files (same rows as informative)."""
    frames = [pd.read_csv(splits_dir / f"task_humanitarian_text_img_{s}.tsv",
                          sep="\t", dtype=str, quoting=3)
              for s in ["train", "dev", "test"]]
    return pd.concat(frames, ignore_index=True)


def image_features(paths, device):
    """Pooled 1280-d EfficientNet-B0 features, one row per image."""
    model = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1")
    model.classifier = torch.nn.Identity()
    model = model.eval().to(device)
    out = []
    with torch.no_grad():
        for batch in DataLoader(Images(paths), batch_size=64):
            out.append(model(batch.to(device)).float().cpu().numpy())
    return np.concatenate(out)


def text_features(texts, device):
    """Mean-pooled 768-d BERT features over real tokens."""
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    bert = AutoModel.from_pretrained("bert-base-uncased").eval().to(device)
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            enc = tok(texts[i:i + 64], padding=True, truncation=True,
                      max_length=96, return_tensors="pt").to(device)
            hidden = bert(**enc).last_hidden_state
            mask = enc.attention_mask.unsqueeze(-1).float()
            pooled = (hidden * mask).sum(1) / mask.sum(1)
            out.append(pooled.cpu().numpy())
    return np.concatenate(out)

In [3]:
data_dir = Path('../data/CrisisMMD_v2.0')
splits_dir = Path('../data/splits')
out_dir = Path('../cache')
out_dir.mkdir(parents=True, exist_ok=True)
device = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cpu')
index = load_index(splits_dir / 'crisismmd_datasplit_all')

t = time.time()
img = image_features([data_dir / p for p in index.image], device)
np.savez(out_dir / 'image_b0.npz', image_id=index.image_id.values,
         feat=img.astype(np.float16))
print(f'images {len(img)} in {time.time() - t:.0f} s')

tweets = index.drop_duplicates('tweet_id')
texts = [clean_text(x) for x in tweets.tweet_text.fillna('')]
t = time.time()
txt = text_features(texts, device)
np.savez(out_dir / 'text_bert.npz', tweet_id=tweets.tweet_id.values,
         feat=txt.astype(np.float16))
print(f'tweets {len(txt)} in {time.time() - t:.0f} s')

images 18082 in 132 s


tweets 16058 in 74 s


## 2. Logistic regression comparison

The regularisation strength C and the late fusion weight are chosen on dev macro-F1. Classes are weighted by inverse frequency. The test set is used once. The bootstrap interval is for the test macro-F1 difference between each fusion model and the unimodal model that is best on dev (1,000 resamples).

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

SPLITS = Path("../data/splits") / "crisismmd_datasplit_agreed_label"
CACHE = Path("../cache")
C_GRID = [0.001, 0.01, 0.1, 1.0]
RNG = np.random.default_rng(0)

In [5]:
def load_features():
    """Image features keyed by image_id and text features keyed by tweet_id."""
    img = np.load(CACHE / "image_b0.npz", allow_pickle=True)
    txt = np.load(CACHE / "text_bert.npz", allow_pickle=True)
    img_map = dict(zip(img["image_id"], range(len(img["image_id"]))))
    txt_map = dict(zip(txt["tweet_id"], range(len(txt["tweet_id"]))))
    return (img["feat"].astype(np.float32), img_map,
            txt["feat"].astype(np.float32), txt_map)


def load_task(task, feats):
    """Feature matrices and labels for the train, dev and test splits."""
    img, img_map, txt, txt_map = feats
    out = {}
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(SPLITS / f"task_{task}_text_img_agreed_lab_{split}"
                         ".tsv", sep="\t", dtype=str, quoting=3)
        out[split] = {
            "image": img[[img_map[i] for i in df.image_id]],
            "text": txt[[txt_map[i] for i in df.tweet_id]],
            "y": df.label.values}
    return out


def fit_best(x_train, y_train, x_dev, y_dev):
    """Fit logistic regression for each C and keep the best on dev macro-F1."""
    scaler = StandardScaler().fit(x_train)
    xt, xd = scaler.transform(x_train), scaler.transform(x_dev)
    best = None
    for c in C_GRID:
        model = LogisticRegression(C=c, max_iter=3000,
                                   class_weight="balanced").fit(xt, y_train)
        score = f1_score(y_dev, model.predict(xd), average="macro")
        if best is None or score > best[0]:
            best = (score, c, model)
    return scaler, best[2], best[1]


def scores(y, pred):
    """Accuracy, macro-F1 and weighted F1."""
    return (accuracy_score(y, pred), f1_score(y, pred, average="macro"),
            f1_score(y, pred, average="weighted"))


def bootstrap_diff(y, pred_a, pred_b, n=1000):
    """95% interval of macro-F1(a) minus macro-F1(b) over test resamples."""
    idx = RNG.integers(0, len(y), size=(n, len(y)))
    diffs = [f1_score(y[i], pred_a[i], average="macro")
             - f1_score(y[i], pred_b[i], average="macro") for i in idx]
    return np.percentile(diffs, [2.5, 97.5])


def run(task, feats):
    """Train the four models on one task and print the comparison."""
    d = load_task(task, feats)
    for s in d.values():
        s["early"] = np.hstack([s["image"], s["text"]])
    rows, preds, probs = [], {}, {}
    for name in ["image", "text", "early"]:
        scaler, model, c = fit_best(d["train"][name], d["train"]["y"],
                                    d["dev"][name], d["dev"]["y"])
        for split in ["train", "dev", "test"]:
            x = scaler.transform(d[split][name])
            probs[name, split] = model.predict_proba(x)
            preds[name, split] = model.predict(x)
        classes = model.classes_
        rows.append([name, c])

    # late fusion: weighted mean of unimodal probabilities, weight on dev
    best_w = max(np.linspace(0, 1, 11), key=lambda w: f1_score(
        d["dev"]["y"], classes[(w * probs["image", "dev"] + (1 - w)
                                * probs["text", "dev"]).argmax(1)],
        average="macro"))
    for split in ["train", "dev", "test"]:
        p = best_w * probs["image", split] + (1 - best_w) * probs[
            "text", split]
        preds["late", split] = classes[p.argmax(1)]
    rows.append(["late", f"w_image={best_w:.1f}"])

    table = []
    for name, setting in rows:
        acc, mf1, wf1 = scores(d["test"]["y"], preds[name, "test"])
        tr = f1_score(d["train"]["y"], preds[name, "train"],
                      average="macro")
        dv = f1_score(d["dev"]["y"], preds[name, "dev"], average="macro")
        table.append([name, setting, round(tr, 3), round(dv, 3),
                      round(acc, 3), round(mf1, 3), round(wf1, 3)])
    print(f"\n=== {task} (agreed labels) train {len(d['train']['y'])}, "
          f"dev {len(d['dev']['y'])}, test {len(d['test']['y'])} ===")
    print(pd.DataFrame(table, columns=[
        "model", "setting", "train mF1", "dev mF1", "test acc",
        "test mF1", "test wF1"]).to_string(index=False))

    y = d["test"]["y"]
    uni = max(["image", "text"], key=lambda m: f1_score(
        d["dev"]["y"], preds[m, "dev"], average="macro"))
    for fused in ["early", "late"]:
        lo, hi = bootstrap_diff(y, preds[fused, "test"], preds[uni, "test"])
        print(f"{fused} minus {uni} test macro-F1, 95% bootstrap interval "
              f"[{lo:.3f}, {hi:.3f}]")
    print("\nper-class test F1")
    per = {m: f1_score(y, preds[m, "test"], average=None, labels=classes)
           for m in ["image", "text", "early", "late"]}
    print(pd.DataFrame(per, index=classes).round(3).to_string())

In [6]:
%%time
feats = load_features()
for task in ["informative", "humanitarian"]:
    run(task, feats)


=== informative (agreed labels) train 9601, dev 1573, test 1534 ===
model     setting  train mF1  dev mF1  test acc  test mF1  test wF1
image       0.001      0.852    0.806     0.842     0.826     0.844
 text       0.001      0.833    0.819     0.845     0.830     0.847
early       0.001      0.916    0.876     0.901     0.890     0.902
 late w_image=0.5      0.906    0.885     0.900     0.889     0.901


early minus text test macro-F1, 95% bootstrap interval [0.041, 0.078]


late minus text test macro-F1, 95% bootstrap interval [0.039, 0.076]

per-class test F1
                 image  text  early   late
informative      0.878  0.88  0.925  0.925
not_informative  0.773  0.78  0.854  0.853



=== humanitarian (agreed labels) train 6126, dev 998, test 955 ===
model     setting  train mF1  dev mF1  test acc  test mF1  test wF1
image       0.001      0.792    0.626     0.781     0.702     0.784
 text       0.001      0.760    0.648     0.758     0.677     0.764
early        0.01      0.981    0.781     0.862     0.794     0.864
 late w_image=0.5      0.877    0.753     0.859     0.802     0.861


early minus text test macro-F1, 95% bootstrap interval [0.071, 0.164]


late minus text test macro-F1, 95% bootstrap interval [0.087, 0.166]

per-class test F1
                                        image   text  early   late
affected_individuals                    0.452  0.438  0.538  0.615
infrastructure_and_utility_damage       0.775  0.687  0.902  0.864
not_humanitarian                        0.813  0.801  0.887  0.882
other_relevant_information              0.803  0.753  0.854  0.861
rescue_volunteering_or_donation_effort  0.667  0.706  0.789  0.789
CPU times: user 27 s, sys: 270 ms, total: 27.3 s
Wall time: 27.4 s


## 3. Reading the results

Train, dev and test macro-F1 are printed side by side, so the gap between train and test shows overfitting. Early fusion on the humanitarian task has the largest gap. The best C sits at the lower edge of the grid for most models, so the full project extends the grid. Near duplicates across splits are still present in this run.